# MovieLens Latest Small — Preprocessing
**Goal:** Filter cold items, remap IDs to dense indices, train/test split, and save clean artefacts.

No rating transformations — ratings are ordinal labels, not features. Missing entries are not filled — they're what we're trying to predict.


## 1. Imports & Config

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── plot style ────────────────────────────────
BG, PANEL, GRID_C = "#0f0f14", "#16161f", "#2a2a35"
TEXT_C, ACCENT, ACCENT2, ACCENT3 = "#d4d4e0", "#7DF9C4", "#F97D7D", "#7DA8F9"

def style_ax(ax, title):
    ax.set_facecolor(PANEL)
    ax.set_title(title, color=TEXT_C, fontsize=10, pad=9, fontweight="bold")
    ax.tick_params(colors=TEXT_C, labelsize=8)
    ax.xaxis.label.set_color(TEXT_C)
    ax.yaxis.label.set_color(TEXT_C)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID_C)
    ax.grid(color=GRID_C, linewidth=0.5, linestyle="--", alpha=0.7)

# ── paths ─────────────────────────────────────
RATINGS_PATH = "../data/raw/ratings.csv"
MOVIES_PATH  = "../data/raw/movies.csv"

# ── preprocessing knobs ───────────────────────
MIN_MOVIE_RATINGS = 10   # drop movies with fewer ratings than this
MIN_USER_RATINGS  = 5    # drop users with fewer ratings than this (already ≥20 here)
TEST_FRAC         = 0.2
SEED              = 42


## 2. Load Raw Data

In [5]:
ratings_raw = pd.read_csv(RATINGS_PATH)
movies_raw  = pd.read_csv(MOVIES_PATH)

ratings_raw["timestamp"] = pd.to_datetime(ratings_raw["timestamp"], unit="s")

print(f"Raw ratings : {len(ratings_raw):,}")
print(f"Raw users   : {ratings_raw.userId.nunique():,}")
print(f"Raw movies  : {ratings_raw.movieId.nunique():,}")
print()
print(ratings_raw.head())


Raw ratings : 100,836
Raw users   : 610
Raw movies  : 9,724

   userId  movieId  rating           timestamp
0       1        1     4.0 2000-07-30 18:45:03
1       1        3     4.0 2000-07-30 18:20:47
2       1        6     4.0 2000-07-30 18:37:04
3       1       47     5.0 2000-07-30 19:03:35
4       1       50     5.0 2000-07-30 18:48:51


## 3. Extract Year from Title

Movie titles in this dataset embed the release year: `"Toy Story (1995)"`.  
We parse it out as a separate column — useful for later analysis and potentially as a feature.


In [6]:
movies = movies_raw.copy()

# Extract year with regex — handles titles like "Toy Story (1995)"
movies["year"] = movies["title"].str.extract(r'\((\d{4})\)$').astype(float)

# Clean title: strip the " (YYYY)" suffix
movies["title_clean"] = movies["title"].str.replace(r'\s*\(\d{4}\)$', '', regex=True).str.strip()

# Parse genres into a list
movies["genre_list"] = movies["genres"].str.split("|")

missing_year = movies["year"].isna().sum()
print(f"Movies with parseable year : {movies.year.notna().sum():,}")
print(f"Movies missing year        : {missing_year}")
print()
print(movies.head())


Movies with parseable year : 9,718
Movies missing year        : 24

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres    year  \
0  Adventure|Animation|Children|Comedy|Fantasy  1995.0   
1                   Adventure|Children|Fantasy  1995.0   
2                               Comedy|Romance  1995.0   
3                         Comedy|Drama|Romance  1995.0   
4                                       Comedy  1995.0   

                   title_clean  \
0                    Toy Story   
1                      Jumanji   
2             Grumpier Old Men   
3            Waiting to Exhale   
4  Father of the Bride Part II   

                                          genre_list  
0  [Adventure, Animati

## 4. Cold-Start Filtering

Drop movies and users below minimum interaction thresholds.  
We apply this **iteratively** — dropping cold movies may make some users cold, and vice versa.  
We loop until the dataset stabilises.


In [7]:
ratings = ratings_raw.copy()

iteration = 0
while True:
    before = len(ratings)

    # Drop cold movies
    movie_counts = ratings.groupby("movieId")["rating"].count()
    valid_movies = movie_counts[movie_counts >= MIN_MOVIE_RATINGS].index
    ratings = ratings[ratings["movieId"].isin(valid_movies)]

    # Drop cold users
    user_counts = ratings.groupby("userId")["rating"].count()
    valid_users = user_counts[user_counts >= MIN_USER_RATINGS].index
    ratings = ratings[ratings["userId"].isin(valid_users)]

    iteration += 1
    after = len(ratings)
    print(f"  Iteration {iteration}: {before:,} → {after:,} ratings")
    if before == after:
        break

print()
print(f"{'':30} {'BEFORE':>10}  {'AFTER':>10}  {'DROPPED':>10}")
print(f"  {'Ratings':<28} {len(ratings_raw):>10,}  {len(ratings):>10,}  {len(ratings_raw)-len(ratings):>10,}")
print(f"  {'Users':<28} {ratings_raw.userId.nunique():>10,}  {ratings.userId.nunique():>10,}  {ratings_raw.userId.nunique()-ratings.userId.nunique():>10,}")
print(f"  {'Movies':<28} {ratings_raw.movieId.nunique():>10,}  {ratings.movieId.nunique():>10,}  {ratings_raw.movieId.nunique()-ratings.movieId.nunique():>10,}")

sparsity_before = 1 - len(ratings_raw) / (ratings_raw.userId.nunique() * ratings_raw.movieId.nunique())
sparsity_after  = 1 - len(ratings)     / (ratings.userId.nunique()      * ratings.movieId.nunique())
print(f"  {'Sparsity':<28} {sparsity_before:>10.3%}  {sparsity_after:>10.3%}")


  Iteration 1: 100,836 → 81,116 ratings
  Iteration 2: 81,116 → 81,116 ratings

                                   BEFORE       AFTER     DROPPED
  Ratings                         100,836      81,116      19,720
  Users                               610         610           0
  Movies                            9,724       2,269       7,455
  Sparsity                        98.300%     94.139%


## 6. Remap IDs to Dense 0-Based Indices

Raw IDs are arbitrary integers (movieId goes up to 193,609).  
We need **dense 0-based indices** to index into the P and Q matrices in our MF model.  
We save the mappings so we can decode predictions back to real movie titles later.


In [9]:
# Build encoders
user_ids  = sorted(ratings["userId"].unique())
movie_ids = sorted(ratings["movieId"].unique())

user_to_idx  = {uid: i for i, uid in enumerate(user_ids)}
movie_to_idx = {mid: i for i, mid in enumerate(movie_ids)}

# Inverse maps — for decoding predictions
idx_to_user  = {i: uid for uid, i in user_to_idx.items()}
idx_to_movie = {i: mid for mid, i in movie_to_idx.items()}

# Apply to ratings
ratings = ratings.copy()
ratings["user_idx"]  = ratings["userId"].map(user_to_idx)
ratings["movie_idx"] = ratings["movieId"].map(movie_to_idx)

n_users  = len(user_to_idx)
n_movies = len(movie_to_idx)

print(f"n_users  = {n_users}")
print(f"n_movies = {n_movies}")
print(f"user_idx  range: {ratings.user_idx.min()} – {ratings.user_idx.max()}")
print(f"movie_idx range: {ratings.movie_idx.min()} – {ratings.movie_idx.max()}")
print()
print(ratings[["userId","user_idx","movieId","movie_idx","rating"]].head(8))


n_users  = 610
n_movies = 2269
user_idx  range: 0 – 609
movie_idx range: 0 – 2268

   userId  user_idx  movieId  movie_idx  rating
0       1         0        1          0     4.0
1       1         0        3          2     4.0
2       1         0        6          4     4.0
3       1         0       47         34     5.0
4       1         0       50         36     5.0
5       1         0       70         43     3.0
6       1         0      101         55     5.0
7       1         0      110         59     4.0


## 7. Train / Test Split

**Per-user holdout**: for each user, we hold out 20% of their ratings as the test set.  

Why not random split? Because a pure random split can put *all* of a user's ratings in test — 
the model would have never seen that user during training (cold start). Per-user split guarantees 
every user appears in both train and test.


In [10]:
rng = np.random.default_rng(SEED)
train_rows, test_rows = [], []

for _, group in ratings.groupby("user_idx"):
    idx = group.index.to_numpy().copy()
    rng.shuffle(idx)
    split = max(1, int(len(idx) * TEST_FRAC))
    test_rows.extend(idx[:split])
    train_rows.extend(idx[split:])

train = ratings.loc[train_rows].reset_index(drop=True)
test  = ratings.loc[test_rows].reset_index(drop=True)

print(f"Train : {len(train):,} ratings")
print(f"Test  : {len(test):,}  ratings")
print(f"Split : {len(test)/(len(train)+len(test)):.1%} test")
print()

# Sanity checks
assert set(test.user_idx) <= set(train.user_idx), "Cold-start users in test!"
print("✓ All test users appear in train")
print(f"✓ Train users: {train.user_idx.nunique()}  |  Test users: {test.user_idx.nunique()}")

# Ratings per user in train vs test
train_rpu = train.groupby("user_idx")["rating"].count()
test_rpu  = test.groupby("user_idx")["rating"].count()
print(f"\nTrain ratings/user — min: {train_rpu.min()}  median: {train_rpu.median():.0f}  max: {train_rpu.max()}")
print(f"Test  ratings/user — min: {test_rpu.min()}   median: {test_rpu.median():.0f}  max: {test_rpu.max()}")


Train : 65,130 ratings
Test  : 15,986  ratings
Split : 19.7% test

✓ All test users appear in train
✓ Train users: 610  |  Test users: 610

Train ratings/user — min: 6  median: 52  max: 1308
Test  ratings/user — min: 1   median: 12  max: 326


## 8. Final Summary & Save Artefacts

In [11]:
print("=" * 50)
print("  Preprocessed Dataset Summary")
print("=" * 50)
print(f"  Users        : {n_users:,}")
print(f"  Movies       : {n_movies:,}")
print(f"  Total ratings: {len(ratings):,}")
print(f"  Sparsity     : {1 - len(ratings)/(n_users*n_movies):.3%}")
print(f"  Rating range : {ratings.rating.min()} – {ratings.rating.max()}")
print(f"  Global mean  : {ratings.rating.mean():.4f}")
print()
print(f"  Train        : {len(train):,}")
print(f"  Test         : {len(test):,}")
print("=" * 50)

# Save
train.to_csv("../data/processed/train.csv", index=False)
test.to_csv("../data/processed/test.csv",   index=False)

# Save the filtered movies table (with year + clean title)
movies_filtered = movies[movies["movieId"].isin(movie_to_idx.keys())].copy()
movies_filtered["movie_idx"] = movies_filtered["movieId"].map(movie_to_idx)
movies_filtered.to_csv("../data/processed/movies_clean.csv", index=False)

# Save ID maps as CSVs (useful for decoding later)
pd.DataFrame({"userId": user_ids,  "user_idx":  range(n_users)}).to_csv("../data/processed/user_map.csv",  index=False)
pd.DataFrame({"movieId": movie_ids, "movie_idx": range(n_movies)}).to_csv("../data/processed/movie_map.csv", index=False)

print("\nSaved: ../data/processed/train.csv, ../data/processed/test.csv, ../data/processed/movies_clean.csv, ../data/processed/user_map.csv, ../data/processed/movie_map.csv")


  Preprocessed Dataset Summary
  Users        : 610
  Movies       : 2,269
  Total ratings: 81,116
  Sparsity     : 94.139%
  Rating range : 0.5 – 5.0
  Global mean  : 3.5737

  Train        : 65,130
  Test         : 15,986

Saved: ../data/processed/train.csv, ../data/processed/test.csv, ../data/processed/movies_clean.csv, ../data/processed/user_map.csv, ../data/processed/movie_map.csv


In [13]:
N_MOVIES = len(movie_to_idx)
# ── Tag Processing ─────────────────────────────────────────────────────
tags_raw = pd.read_csv("../data/raw/tags.csv")

# ── Clean ──────────────────────────────────────────────────────────────
tags_raw["tag"] = (tags_raw["tag"]
                   .str.lower()
                   .str.strip()
                   .str.replace("-", " ", regex=False)   # post-apocalyptic → post apocalyptic
                   .str.replace(r"\s+", " ", regex=True) # collapse multiple spaces
                  )

# ── Filter noise tags ──────────────────────────────────────────────────
NOISE_TAGS = {
    "in netflix queue", "netflix queue",
    "not available from netflix", "no dvd at netflix",
    "free to download", "seen", "seen at the cinema",
    "seen more than once", "not seen", "watched",
    "good", "bad", "great movie", "really bad",
    "awesome", "amazing", "best picture",
    "masterpiece", "classic movie", "imdb top 250",
    "test tag", "for katie", "ummarti2006",
}

tags_clean = tags_raw[~tags_raw["tag"].isin(NOISE_TAGS)].copy()

# ── Keep only tags for movies in our filtered set ──────────────────────
tags_clean = tags_clean[tags_clean["movieId"].isin(movie_to_idx)].copy()
tags_clean["movie_idx"] = tags_clean["movieId"].map(movie_to_idx)

# ── Summary ────────────────────────────────────────────────────────────
print("=" * 50)
print("  Tag Processing Summary")
print("=" * 50)
print(f"  Raw tags             : {len(tags_raw):,}")
print(f"  After noise filter   : {len(tags_clean[tags_clean.movieId.isin(movie_to_idx)]):,}")
print(f"  After movie filter   : {len(tags_clean):,}")
print(f"  Movies with tags     : {tags_clean.movie_idx.nunique()} / {N_MOVIES}")
print(f"  Unique tags          : {tags_clean.tag.nunique():,}")
print(f"  Tags per movie       : min={tags_clean.groupby('movie_idx').size().min()}  "
      f"median={tags_clean.groupby('movie_idx').size().median():.0f}  "
      f"max={tags_clean.groupby('movie_idx').size().max()}")
print(f"  Movies with no tags  : {N_MOVIES - tags_clean.movie_idx.nunique()}")
print("=" * 50)

# ── Save ───────────────────────────────────────────────────────────────
tags_clean.to_csv("../data/processed/tags_clean.csv", index=False)
print("\nSaved: tags_clean.csv")
print(tags_clean[["movie_idx", "movieId", "tag"]].head(10))

  Tag Processing Summary
  Raw tags             : 3,683
  After noise filter   : 2,650
  After movie filter   : 2,650
  Movies with tags     : 896 / 2269
  Unique tags          : 1,177
  Tags per movie       : min=1  median=1  max=176
  Movies with no tags  : 1373

Saved: tags_clean.csv
   movie_idx  movieId                tag
0       1952    60756              funny
1       1952    60756    highly quotable
2       1952    60756       will ferrell
3       2096    89774       boxing story
4       2096    89774                mma
5       2096    89774          tom hardy
6       2172   106782              drugs
7       2172   106782  leonardo dicaprio
8       2172   106782    martin scorsese
9       1840    48516       way too long
